# Parte 5: Online Retail

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np

from src.limpieza import limpiar     # la misma limpieza de la tarea de series de tiempo

# InvoiceNo y StockCode mezclan números y texto: se leen como texto
raw = pd.read_excel("../data/raw/Online_Retail.xlsx", dtype={"InvoiceNo": str, "StockCode": str})

In [ ]:
df, bitacora = limpiar(raw, devolver_bitacora=True)
print(bitacora.to_string(index=False))

df = df.dropna(subset=["CustomerID"])      # el BG/NBD necesita identificar al cliente
print(df.shape, df["CustomerID"].nunique())

In [ ]:
df["fecha"] = pd.to_datetime(df["InvoiceDate"]).dt.normalize()
df["venta"] = df["Quantity"] * df["UnitPrice"]

# Transacción = día con venta neta positiva de un cliente
tx_dia = (df.groupby(["CustomerID", "fecha"])["venta"].sum()
            .reset_index().query("venta > 0"))

fin_cal  = pd.Timestamp("2011-06-30")
ini_hold = pd.Timestamp("2011-07-01")
fin_hold = pd.Timestamp("2011-12-08")
tau = (fin_hold - fin_cal).days / 7          # semanas de holdout

cal  = tx_dia[tx_dia.fecha <= fin_cal]
hold = tx_dia[(tx_dia.fecha >= ini_hold) & (tx_dia.fecha <= fin_hold)]

g = cal.groupby("CustomerID")["fecha"]
rfm = pd.DataFrame({
    "x":  g.count() - 1,
    "tx": (g.max() - g.min()).dt.days / 7,
    "T":  (fin_cal - g.min()).dt.days / 7,
})
rfm["x_star"] = hold.groupby("CustomerID").size().reindex(rfm.index, fill_value=0)

print("Clientes en la cohorte:", len(rfm))
print("Semanas de holdout:", round(tau, 1))
print("% sin compras repetidas:", round((rfm.x == 0).mean() * 100, 1))
print("Compras reales en holdout:", rfm.x_star.sum())
rfm.describe().round(2)

## 5.1 Ajuste del BG/NBD.

In [ ]:
from scipy.special import gammaln, betaln
from scipy.optimize import minimize

x, tx, T = rfm["x"].values, rfm["tx"].values, rfm["T"].values

def neg_ll(log_params, x, tx, T):
    r, alpha, a, b = np.exp(log_params)
    x1 = np.maximum(x, 1)                                   # evita betaln(a+1, b-1) cuando x = 0
    t1 = betaln(a, b + x) - betaln(a, b) - (r + x) * np.log(alpha + T)
    t2 = np.where(x > 0,
                  betaln(a + 1, b + x1 - 1) - betaln(a, b) - (r + x) * np.log(alpha + tx),
                  -np.inf)
    ll = gammaln(r + x) - gammaln(r) + r * np.log(alpha) + np.logaddexp(t1, t2)
    return -ll.sum()

# Varios puntos de arranque
inicios = [[0, 0, 0, 0], [np.log(0.25), np.log(4), np.log(0.8), np.log(2.5)], [1, 2, -1, 1]]
resultados = []
for p0 in inicios:
    res = minimize(neg_ll, p0, args=(x, tx, T), method="L-BFGS-B", bounds=[(-10, 10)] * 4)
    resultados.append(res)
    print(np.round(np.exp(res.x), 4), round(-res.fun, 2))

mejor = min(resultados, key=lambda r: r.fun)
r_, alpha_, a_, b_ = np.exp(mejor.x)
print("\nr =", round(r_, 4), " alpha =", round(alpha_, 4), " a =", round(a_, 4), " b =", round(b_, 4))
print("Log-verosimilitud:", round(-mejor.fun, 2))
print("P(morir tras una compra) promedio = a/(a+b) =", round(a_ / (a_ + b_), 6))

El BG/NBD estima r = 0.74, α = 9.16, a ≈ 0 y b muy grande (cambia según el punto de arranque, siempre con la misma log-verosimilitud de −15,107.24). La verosimilitud es plana: lo único que los datos identifican es que p = a/(a+b) ≈ 0, es decir, **el modelo concluye que los clientes no se mueren** y colapsa al NBD.

Pasa aquí y no en CDNOW porque: (1) muchos clientes son negocios que compran inventario de forma recurrente; (2) la calibración dura solo ~30 semanas, muy poco para ver silencios largos; (3) los datos empiezan en diciembre de 2010, así que la "primera compra" de muchos clientes no es la real y nos perdemos las muertes tempranas.

## 5.2 Pronóstico del holdout

In [ ]:
from scipy.special import hyp2f1
import matplotlib.pyplot as plt

def E_Y(tau, x, tx, T, r, alpha, a, b):
    A = (alpha + T) / (alpha + T + tau)
    D = 1 + np.where(x > 0, a / (b + np.maximum(x, 1) - 1) * ((alpha + T) / (alpha + tx)) ** (r + x), 0)
    num = (a + b + x - 1) / (a - 1) * (1 - A ** (r + x) * hyp2f1(r + x, b + x, a + b + x - 1, 1 - A))
    return num / D

# Total del holdout
pred = E_Y(tau, x, tx, T, r_, alpha_, a_, b_)
real = rfm["x_star"].sum()
print("Pronóstico BG/NBD:", round(pred.sum()), " Real:", real,
      " Error %:", round((pred.sum() - real) / real * 100, 1))

# Serie semanal: real contra pronóstico
semanas = np.arange(1, 24)
pron_sem = np.diff([E_Y(w, x, tx, T, r_, alpha_, a_, b_).sum() for w in np.r_[0, semanas]])

h = hold[hold.CustomerID.isin(rfm.index)].copy()
h["semana"] = (h.fecha - ini_hold).dt.days // 7 + 1
real_sem = h.groupby("semana").size().reindex(semanas, fill_value=0)

plt.figure(figsize=(9, 4))
plt.plot(semanas, real_sem.values, marker="o", label="Real")
plt.plot(semanas, pron_sem, marker="s", label="BG/NBD")
plt.xticks(semanas[::2], [(ini_hold + pd.Timedelta(weeks=int(s) - 1)).strftime("%d-%b") for s in semanas[::2]], rotation=45)
plt.ylabel("Transacciones por semana")
plt.title("Holdout: julio a diciembre 2011")
plt.legend(); plt.tight_layout(); plt.show()

El BG/NBD pronostica 5,489 transacciones en el holdout contra 6,313 reales (error de −13.1%). Con p ≈ 0 el pronóstico semanal es prácticamente constante (~240 por semana). Acierta de julio a mediados de octubre, pero desde finales de octubre la demanda real sube a ~380–400 por semana: es el pico de noviembre que vimos en la tarea de series de tiempo, cuando los clientes mayoristas se surten para Navidad. El modelo supone compras estacionarias y se calibró con datos de diciembre a junio, así que no puede anticipar esa estacionalidad. **El error no es de nivel, es de estacionalidad.**